In [ ]:
!pip install lifelines

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 15.1 MB/s eta 0:00:00
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4030 sha256=ab0b56f91b523b322260a81fa80c3b7ca7d1a390089e14903a6fade16f681882
  Stored in directory: /root/.cache/pip/wheels/50/37/21/0a719b9d89c635e89ff24bd93b862882ad675279552013b2fb
Successfully built autograd-gamma


In [ ]:
import os, warnings, pickle, joblib, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier, VotingClassifier
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      cross_val_score, RandomizedSearchCV)
from sklearn.preprocessing   import StandardScaler, LabelEncoder
from sklearn.impute           import SimpleImputer
from sklearn.metrics          import (roc_auc_score, average_precision_score,
                                      f1_score, classification_report,
                                      confusion_matrix, roc_curve,
                                      precision_recall_curve, ConfusionMatrixDisplay)
from sklearn.calibration      import calibration_curve
from sklearn.pipeline         import Pipeline

# Boosting
import xgboost  as xgb
import lightgbm as lgb

# Survival
from lifelines import CoxPHFitter, KaplanMeierFitter

# Imbalanced
from imblearn.over_sampling import SMOTE

# SHAP
import shap

print("✅ All libraries loaded")
print(f"   XGBoost  : {xgb.__version__}")
print(f"   LightGBM : {lgb.__version__}")

# ── Paths ──────────────────────────────────────────────────
# ★ EDIT THESE TWO LINES to match where you extracted your datasets
MIMIC_PATH  = '/content/drive/MyDrive/DataScience-Project/mimic-iv-clinical-database-demo-2.2'
VITALS_PATH = '/content/drive/MyDrive/DataScience-Project/human-vitals-dataset/human_vital_signs_dataset_2024.csv'

# Output folders (auto-created)
BASE_OUT   = '/content/drive/MyDrive/CUEPE_Outputs'
PLOT_DIR   = os.path.join(BASE_OUT, 'plots')
MODEL_DIR  = os.path.join(BASE_OUT, 'models')
RESULT_DIR = os.path.join(BASE_OUT, 'results')
for d in [PLOT_DIR, MODEL_DIR, RESULT_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"\n📁 Outputs → {BASE_OUT}")

# ── Plot style ─────────────────────────────────────────────
DARK_BG  = '#0d1117'
BLUE     = '#3b82f6'
GREEN    = '#10b981'
AMBER    = '#f59e0b'
RED      = '#ef4444'
PURPLE   = '#8b5cf6'
TEAL     = '#06b6d4'
COLORS   = [BLUE, GREEN, AMBER, RED, PURPLE, TEAL, '#f472b6', '#a3e635']

plt.rcParams.update({
    'figure.facecolor' : DARK_BG,
    'axes.facecolor'   : '#161b22',
    'axes.edgecolor'   : '#30363d',
    'axes.labelcolor'  : '#c9d1d9',
    'xtick.color'      : '#8b949e',
    'ytick.color'      : '#8b949e',
    'text.color'       : '#c9d1d9',
    'grid.color'       : '#21262d',
    'grid.linestyle'   : '--',
    'legend.facecolor' : '#161b22',
    'legend.edgecolor' : '#30363d',
    'font.family'      : 'monospace',
    'font.size'        : 10,
})

def savefig(name):
    path = os.path.join(PLOT_DIR, name)
    plt.savefig(path, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.show()
    print(f"  📊 Saved: {name}")

✅ All libraries loaded
   XGBoost  : 3.2.0
   LightGBM : 4.6.0

📁 Outputs → /content/drive/MyDrive/CUEPE_Outputs


In [ ]:
print("\n" + "═"*60)
print("  SECTION 1 · Data Loading")
print("═"*60)

# ── 1A · MIMIC-IV tables ──────────────────────────────────
patients   = pd.read_csv(os.path.join(MIMIC_PATH, 'hosp/patients.csv'))
icustays   = pd.read_csv(os.path.join(MIMIC_PATH, 'icu/icustays.csv'))
admissions = pd.read_csv(os.path.join(MIMIC_PATH, 'hosp/admissions.csv'))

# Chartevents — only vital columns to save memory
VITAL_ITEMIDS = {
    220045: 'heart_rate',
    220179: 'sbp',
    220180: 'dbp',
    220210: 'resp_rate',
    223762: 'temp_c',
    220277: 'spo2',
    220739: 'gcs_eye',
    223900: 'gcs_verbal',
    223901: 'gcs_motor',
}
ce_raw = pd.read_csv(
    os.path.join(MIMIC_PATH, 'icu/chartevents.csv'),
    usecols=['subject_id', 'stay_id', 'charttime', 'itemid', 'valuenum']
)
ce = ce_raw[ce_raw['itemid'].isin(VITAL_ITEMIDS)].copy()
ce['vital']     = ce['itemid'].map(VITAL_ITEMIDS)
ce['charttime'] = pd.to_datetime(ce['charttime'])
ce = ce.dropna(subset=['valuenum'])

# Labevents — key chemistry / haematology panels
KEY_LABS = {
    50912: 'creatinine', 51006: 'bun',       50931: 'glucose',
    51222: 'hemoglobin', 51265: 'platelets',  50971: 'potassium',
    50983: 'sodium',     50882: 'bicarbonate',50868: 'anion_gap',
}
labs_raw = pd.read_csv(
    os.path.join(MIMIC_PATH, 'hosp/labevents.csv'),
    usecols=['subject_id', 'itemid', 'valuenum', 'charttime']
)
labs = labs_raw[labs_raw['itemid'].isin(KEY_LABS)].copy()
labs['lab']      = labs['itemid'].map(KEY_LABS)
labs['charttime']= pd.to_datetime(labs['charttime'])
labs = labs.dropna(subset=['valuenum'])

# ── 1B · External vitals dataset ─────────────────────────
vitals_ext = pd.read_csv(VITALS_PATH)
vitals_ext['Gender'] = LabelEncoder().fit_transform(vitals_ext['Gender'].astype(str))

print(f"  patients    : {patients.shape}")
print(f"  icustays    : {icustays.shape}")
print(f"  chartevents : {ce.shape}  (vitals only)")
print(f"  labevents   : {labs.shape}")
print(f"  vitals_ext  : {vitals_ext.shape}")
print("✅ Data loaded")


════════════════════════════════════════════════════════════
  SECTION 1 · Data Loading
════════════════════════════════════════════════════════════
  patients    : (100, 6)
  icustays    : (140, 8)
  chartevents : (68244, 6)  (vitals only)
  labevents   : (26041, 5)
  vitals_ext  : (200020, 17)
✅ Data loaded


In [ ]:
print("\n" + "═"*60)
print("  SECTION 2 · Feature Engineering")
print("═"*60)

def uncertainty_features(values, prefix):
    """Compute per-signal uncertainty metrics."""
    vals = np.array(values)
    if len(vals) < 3:
        return {}
    mu   = np.mean(vals)
    sig  = np.std(vals)
    # Differential entropy via histogram
    h, _ = np.histogram(vals, bins=min(10, len(vals)))
    p    = h / (h.sum() + 1e-9)
    entr = -np.sum(p * np.log(p + 1e-9))
    # Linear trend slope (proxy for direction of drift)
    slope = np.polyfit(np.arange(len(vals)), vals, 1)[0]
    return {
        f'{prefix}_mean'    : mu,
        f'{prefix}_std'     : sig,
        f'{prefix}_cv'      : sig / (abs(mu) + 1e-6),        # coefficient of variation
        f'{prefix}_range'   : vals.max() - vals.min(),
        f'{prefix}_iqr'     : np.percentile(vals, 75) - np.percentile(vals, 25),
        f'{prefix}_entropy' : entr,
        f'{prefix}_slope'   : slope,                          # temporal drift
        f'{prefix}_n'       : len(vals),
    }

# ── 2A · Vital sign features per patient ─────────────────
print("⏳ Computing vital features …")
vframes = []
for (sid, vname), grp in ce.groupby(['subject_id', 'vital']):
    feats = uncertainty_features(grp['valuenum'].values, vname)
    if feats:
        feats['subject_id'] = sid
        vframes.append(feats)

vfeat = pd.DataFrame(vframes).groupby('subject_id').first().reset_index()
print(f"  Vital feature matrix: {vfeat.shape}")

# ── 2B · Lab features per patient ────────────────────────
print("⏳ Computing lab features …")
lframes = []
for (sid, lname), grp in labs.groupby(['subject_id', 'lab']):
    feats = uncertainty_features(grp['valuenum'].values, f'lab_{lname}')
    if feats:
        feats['subject_id'] = sid
        lframes.append(feats)

lfeat = pd.DataFrame(lframes).groupby('subject_id').first().reset_index()
print(f"  Lab feature matrix  : {lfeat.shape}")

# ── 2C · ICU stay meta-features ──────────────────────────
icu = icustays.copy()
icu['intime']  = pd.to_datetime(icu['intime'])
icu['outtime'] = pd.to_datetime(icu['outtime'])
icu['los_hours'] = (icu['outtime'] - icu['intime']).dt.total_seconds() / 3600

icu_feat = icu.groupby('subject_id').agg(
    los_hours     = ('los_hours', 'mean'),
    n_icu_stays   = ('stay_id',   'count'),
    max_los       = ('los_hours', 'max'),
).reset_index()

# ── 2D · Demographics ────────────────────────────────────
pat = patients[['subject_id', 'gender', 'anchor_age']].copy()
pat['gender_enc'] = (pat['gender'] == 'M').astype(int)

# ── 2E · Hospital mortality flag ─────────────────────────
adm_feat = admissions.groupby('subject_id').agg(
    hospital_expire_flag = ('hospital_expire_flag', 'max'),
    n_admissions         = ('hadm_id', 'count'),
).reset_index()

# ── 2F · Merge everything ────────────────────────────────
df = pat.merge(icu_feat,  on='subject_id', how='inner')
df = df.merge(vfeat,      on='subject_id', how='left')
df = df.merge(lfeat,      on='subject_id', how='left')
df = df.merge(adm_feat,   on='subject_id', how='left')
df = df.drop(columns=['gender'], errors='ignore')
print(f"\n  Merged MIMIC feature matrix: {df.shape}")


════════════════════════════════════════════════════════════
  SECTION 2 · Feature Engineering
════════════════════════════════════════════════════════════
⏳ Computing vital features …
  Vital feature matrix: (100, 73)
⏳ Computing lab features …
  Lab feature matrix  : (99, 73)

  Merged MIMIC feature matrix: (100, 152)


In [ ]:
print("\n" + "═"*60)
print("  SECTION 3 · Uncertainty Label Construction")
print("═"*60)

# Clinical Uncertainty Index (CUI) – multi-signal composite
# Normalise each component to [0,1] then weight

def safe_normalise(series):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - mn) / (mx - mn)

entropy_cols = [c for c in df.columns if c.endswith('_entropy')]
cv_cols      = [c for c in df.columns if c.endswith('_cv')]
range_cols   = [c for c in df.columns if c.endswith('_range')]
slope_cols   = [c for c in df.columns if c.endswith('_slope')]

# Component 1: Mean prediction entropy across signals
if entropy_cols:
    df['comp_entropy'] = df[entropy_cols].mean(axis=1)
else:
    df['comp_entropy'] = 0.0

# Component 2: Mean coefficient of variation
if cv_cols:
    df['comp_cv'] = df[cv_cols].mean(axis=1)
else:
    df['comp_cv'] = 0.0

# Component 3: Mean range (signal instability)
if range_cols:
    df['comp_range'] = df[range_cols].mean(axis=1)
else:
    df['comp_range'] = 0.0

# Component 4: Mean absolute slope (temporal drift)
if slope_cols:
    df['comp_drift'] = df[slope_cols].abs().mean(axis=1)
else:
    df['comp_drift'] = 0.0

# Weighted CUI
df['cui_raw'] = (
    0.35 * safe_normalise(df['comp_entropy']) +
    0.25 * safe_normalise(df['comp_cv'])      +
    0.20 * safe_normalise(df['comp_range'])   +
    0.20 * safe_normalise(df['comp_drift'])
)

# Binary label: top 40% CUI → high uncertainty
threshold = df['cui_raw'].quantile(0.60)
df['high_uncertainty'] = (df['cui_raw'] >= threshold).astype(int)

print(f"  CUI threshold (60th pctl): {threshold:.4f}")
print(f"  High-uncertainty patients : {df['high_uncertainty'].sum()}  "
      f"({df['high_uncertainty'].mean()*100:.1f}%)")
print(f"  Low-uncertainty  patients : {(1-df['high_uncertainty']).sum()}")


════════════════════════════════════════════════════════════
  SECTION 3 · Uncertainty Label Construction
════════════════════════════════════════════════════════════
  CUI threshold (60th pctl): 0.4700
  High-uncertainty patients : 40  (40.0%)
  Low-uncertainty  patients : 60


In [ ]:
print("\n" + "═"*60)
print("  SECTION 4 · Exploratory Data Analysis")
print("═"*60)

# ── Plot 1 · CUI Distribution ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Clinical Uncertainty Index (CUI) — Distribution', fontweight='bold')

ax = axes[0]
ax.hist(df[df['high_uncertainty']==0]['cui_raw'], bins=25,
        color=BLUE, alpha=0.7, label='Low Uncertainty', density=True)
ax.hist(df[df['high_uncertainty']==1]['cui_raw'], bins=25,
        color=RED,  alpha=0.7, label='High Uncertainty', density=True)
ax.axvline(threshold, color=AMBER, lw=1.5, ls='--', label=f'Threshold={threshold:.3f}')
ax.set_xlabel('CUI Score'); ax.set_ylabel('Density')
ax.set_title('CUI Score Distribution'); ax.legend()

ax = axes[1]
comp_data = {
    'Entropy\n(w=0.35)' : df['comp_entropy'],
    'CV\n(w=0.25)'      : df['comp_cv'],
    'Range\n(w=0.20)'   : df['comp_range'],
    'Drift\n(w=0.20)'   : df['comp_drift'],
}
bp = ax.boxplot(comp_data.values(), labels=comp_data.keys(),
                patch_artist=True, notch=True)
for patch, color in zip(bp['boxes'], [BLUE, GREEN, AMBER, PURPLE]):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_title('CUI Component Distributions')
ax.set_ylabel('Raw Score')
plt.tight_layout(); savefig('01_cui_distribution.png')

# ── Plot 2 · Vital Sign Entropy by Uncertainty Class ─────
entropy_vitals = [c for c in entropy_cols if c.startswith(('heart','sbp','dbp','resp','spo2'))]
if len(entropy_vitals) >= 3:
    fig, axes = plt.subplots(1, min(5, len(entropy_vitals)), figsize=(16, 4))
    if len(entropy_vitals) == 1: axes = [axes]
    fig.suptitle('Vital Sign Entropy — High vs Low Uncertainty', fontweight='bold')
    for ax, col in zip(axes, entropy_vitals[:5]):
        lo = df[df['high_uncertainty']==0][col].dropna()
        hi = df[df['high_uncertainty']==1][col].dropna()
        ax.hist(lo, bins=15, color=BLUE,  alpha=0.7, density=True, label='Low')
        ax.hist(hi, bins=15, color=RED,   alpha=0.7, density=True, label='High')
        ax.set_title(col.replace('_entropy','').replace('_',' ').title())
        ax.set_xlabel('Entropy'); ax.legend(fontsize=7)
    plt.tight_layout(); savefig('02_vital_entropy_by_class.png')

# ── Plot 3 · Correlation Heatmap ─────────────────────────
num_cols = [c for c in df.select_dtypes(include=np.number).columns
            if c not in ['subject_id','high_uncertainty','cui_raw',
                         'comp_entropy','comp_cv','comp_range','comp_drift']]
corr_cols = num_cols[:20]  # top 20 for readability
corr_df   = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_df, dtype=bool))
sns.heatmap(corr_df, mask=mask, annot=False, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax,
            linewidths=0.3, cbar_kws={'shrink':.7})
ax.set_title('Feature Correlation Matrix (Top 20 Features)', fontweight='bold', pad=12)
plt.tight_layout(); savefig('03_correlation_heatmap.png')

# ── Plot 4 · ICU LOS by uncertainty class ────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('ICU Stay Duration vs Uncertainty', fontweight='bold')

ax = axes[0]
for label, color, name in [(0, BLUE, 'Low Uncertainty'), (1, RED, 'High Uncertainty')]:
    data = df[df['high_uncertainty']==label]['los_hours'].dropna()
    ax.hist(data, bins=20, color=color, alpha=0.7, label=f'{name} (n={len(data)})', density=True)
ax.set_xlabel('ICU LOS (hours)'); ax.set_ylabel('Density')
ax.set_title('LOS Distribution'); ax.legend()

ax = axes[1]
groups = [df[df['high_uncertainty']==0]['los_hours'].dropna(),
          df[df['high_uncertainty']==1]['los_hours'].dropna()]
bp = ax.boxplot(groups, labels=['Low Uncertainty','High Uncertainty'],
                patch_artist=True, notch=True)
bp['boxes'][0].set_facecolor(BLUE); bp['boxes'][0].set_alpha(0.7)
bp['boxes'][1].set_facecolor(RED);  bp['boxes'][1].set_alpha(0.7)
ax.set_ylabel('LOS (hours)'); ax.set_title('LOS Box Plot')
plt.tight_layout(); savefig('04_los_by_uncertainty.png')

# ── Plot 5 · Kaplan-Meier ─────────────────────────────────
try:
    df_surv = df.copy()
    df_surv['duration'] = df_surv['los_hours'].fillna(df_surv['los_hours'].median())
    df_surv = df_surv[df_surv['duration'] > 0]

    kmf = KaplanMeierFitter()
    fig, ax = plt.subplots(figsize=(10, 5))

    for label, color, name in [(0, BLUE, 'Low Uncertainty'), (1, RED, 'High Uncertainty')]:
        sub = df_surv[df_surv['high_uncertainty'] == label]
        kmf.fit(sub['duration'], event_observed=sub['high_uncertainty'], label=name)
        kmf.plot_survival_function(ax=ax, ci_show=True, ci_alpha=0.15, color=color)

    ax.set_title('Kaplan-Meier: Time to Uncertainty Escalation', fontweight='bold')
    ax.set_xlabel('ICU Hours'); ax.set_ylabel('Survival Probability')
    ax.grid(True, alpha=0.3)
    plt.tight_layout(); savefig('05_kaplan_meier.png')
except Exception as e:
    print(f"  ⚠️  KM plot skipped: {e}")

print("✅ EDA plots saved")


════════════════════════════════════════════════════════════
  SECTION 4 · Exploratory Data Analysis
════════════════════════════════════════════════════════════
  📊 Saved: 01_cui_distribution.png
  📊 Saved: 02_vital_entropy_by_class.png
  📊 Saved: 03_correlation_heatmap.png
  📊 Saved: 04_los_by_uncertainty.png
  📊 Saved: 05_kaplan_meier.png
✅ EDA plots saved


In [ ]:
print("\n" + "═"*60)
print("  SECTION 5 · External Vitals Dataset — Feature Engineering")
print("═"*60)

# This 200k-row dataset has a 'Risk Category' label → use it
# as an additional training signal for uncertainty detection.

v = vitals_ext.copy()
v['high_uncertainty'] = (v['Risk Category'] == 'High Risk').astype(int)
v = v.drop(columns=['Risk Category', 'Patient ID', 'Timestamp'], errors='ignore')

# Derived uncertainty features
v['hr_rr_ratio']   = v['Heart Rate']           / (v['Respiratory Rate'] + 1e-6)
v['pulse_pressure']= v['Systolic Blood Pressure'] - v['Diastolic Blood Pressure']
v['spo2_deficit']  = 100 - v['Oxygen Saturation']
v['temp_deviation']= (v['Body Temperature'] - 37.0).abs()

print(f"  External vitals features : {v.shape}")
print(f"  Class balance: High={v['high_uncertainty'].mean()*100:.1f}%  "
      f"Low={(1-v['high_uncertainty'].mean())*100:.1f}%")



════════════════════════════════════════════════════════════
  SECTION 5 · External Vitals Dataset — Feature Engineering
════════════════════════════════════════════════════════════
  External vitals features : (200020, 19)
  Class balance: High=52.6%  Low=47.4%


In [ ]:
print("\n" + "═"*60)
print("  SECTION 6 · MIMIC Dataset — Model Preparation")
print("═"*60)

DROP_COLS = ['subject_id', 'high_uncertainty', 'cui_raw',
             'comp_entropy', 'comp_cv', 'comp_range', 'comp_drift',
             'gender_enc']  # will add back below

feature_cols = [c for c in df.columns
                if c not in ['subject_id','high_uncertainty','cui_raw',
                              'comp_entropy','comp_cv','comp_range','comp_drift']]

X_mimic = df[feature_cols].copy()
y_mimic = df['high_uncertainty'].copy()

# Impute → scale
imputer = SimpleImputer(strategy='median')
scaler  = StandardScaler()
X_imp   = pd.DataFrame(imputer.fit_transform(X_mimic), columns=feature_cols)
X_sc    = pd.DataFrame(scaler.fit_transform(X_imp),    columns=feature_cols)

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_sc, y_mimic, test_size=0.25, random_state=42, stratify=y_mimic)

# SMOTE for class balance
smote = SMOTE(random_state=42)
X_bal_m, y_bal_m = smote.fit_resample(X_train_m, y_train_m)
print(f"  MIMIC train (balanced): {X_bal_m.shape}  test: {X_test_m.shape}")
print(f"  Class balance after SMOTE: {y_bal_m.mean()*100:.1f}% positive")

# Save preprocessors
joblib.dump(imputer, os.path.join(MODEL_DIR, 'imputer_mimic.pkl'))
joblib.dump(scaler,  os.path.join(MODEL_DIR, 'scaler_mimic.pkl'))
joblib.dump(feature_cols, os.path.join(MODEL_DIR, 'feature_cols_mimic.pkl'))
print("  Preprocessors saved ✓")


════════════════════════════════════════════════════════════
  SECTION 6 · MIMIC Dataset — Model Preparation
════════════════════════════════════════════════════════════
  MIMIC train (balanced): (90, 151)  test: (25, 151)
  Class balance after SMOTE: 50.0% positive
  Preprocessors saved ✓


In [ ]:
print("\n" + "═"*60)
print("  SECTION 7 · External Vitals — Model Preparation")
print("═"*60)

feat_v = [c for c in v.columns if c != 'high_uncertainty']
X_v    = v[feat_v].copy()
y_v    = v['high_uncertainty'].copy()

imp_v  = SimpleImputer(strategy='median')
sc_v   = StandardScaler()
Xv_imp = pd.DataFrame(imp_v.fit_transform(X_v), columns=feat_v)
Xv_sc  = pd.DataFrame(sc_v.fit_transform(Xv_imp), columns=feat_v)

X_train_v, X_test_v, y_train_v, y_test_v = train_test_split(
    Xv_sc, y_v, test_size=0.20, random_state=42, stratify=y_v)

smote_v = SMOTE(random_state=42)
X_bal_v, y_bal_v = smote_v.fit_resample(X_train_v, y_train_v)
print(f"  Vitals train (balanced): {X_bal_v.shape}  test: {X_test_v.shape}")

joblib.dump(imp_v,  os.path.join(MODEL_DIR, 'imputer_vitals.pkl'))
joblib.dump(sc_v,   os.path.join(MODEL_DIR, 'scaler_vitals.pkl'))
joblib.dump(feat_v, os.path.join(MODEL_DIR, 'feature_cols_vitals.pkl'))


════════════════════════════════════════════════════════════
  SECTION 7 · External Vitals — Model Preparation
════════════════════════════════════════════════════════════
  Vitals train (balanced): (168184, 18)  test: (40004, 18)


['/content/drive/MyDrive/CUEPE_Outputs/models/feature_cols_vitals.pkl']

In [ ]:
print("\n" + "═"*60)
print("  SECTION 8 · Model Training")
print("═"*60)

results = {}   # {model_name: {auc, ap, f1, y_prob, y_pred, dataset}}

def evaluate(name, model, X_test, y_test, dataset='MIMIC'):
    """Fit already done; just evaluate and store."""
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    auc = roc_auc_score(y_test, y_prob)
    ap  = average_precision_score(y_test, y_prob)
    f1  = f1_score(y_test, y_pred)
    results[name] = dict(auc=auc, ap=ap, f1=f1,
                         y_prob=y_prob, y_pred=y_pred,
                         y_test=np.array(y_test), dataset=dataset)
    print(f"  {name:35s} AUC={auc:.4f}  AP={ap:.4f}  F1={f1:.4f}")
    return auc, f1

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── 8A · Logistic Regression (MIMIC) ──────────────────────
print("\n[MIMIC] Logistic Regression …")
lr = LogisticRegression(C=0.5, max_iter=1000, random_state=42)
lr.fit(X_bal_m, y_bal_m)
evaluate('LR (MIMIC)', lr, X_test_m, y_test_m, 'MIMIC')
cv_lr = cross_val_score(lr, X_sc, y_mimic, cv=cv5, scoring='roc_auc').mean()
print(f"  CV-5 AUC: {cv_lr:.4f}")
joblib.dump(lr, os.path.join(MODEL_DIR, 'lr_mimic.pkl'))

# ── 8B · Random Forest (MIMIC) ───────────────────────────
print("\n[MIMIC] Random Forest …")
rf_param = dict(n_estimators=[200,400], max_depth=[None,10,15],
                min_samples_split=[2,5], max_features=['sqrt','log2'])
rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_rs   = RandomizedSearchCV(rf_base, rf_param, n_iter=8, cv=cv5,
                              scoring='roc_auc', random_state=42, n_jobs=-1)
rf_rs.fit(X_bal_m, y_bal_m)
rf = rf_rs.best_estimator_
evaluate('RF (MIMIC)', rf, X_test_m, y_test_m, 'MIMIC')
print(f"  Best params: {rf_rs.best_params_}")
joblib.dump(rf, os.path.join(MODEL_DIR, 'rf_mimic.pkl'))

# ── 8C · XGBoost (MIMIC) ─────────────────────────────────
print("\n[MIMIC] XGBoost …")
xgb_param = dict(n_estimators=[200,400], max_depth=[4,6],
                  learning_rate=[0.05,0.1], subsample=[0.8,1.0],
                  colsample_bytree=[0.7,1.0])
xgb_base = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                               random_state=42, n_jobs=-1)
xgb_rs   = RandomizedSearchCV(xgb_base, xgb_param, n_iter=8, cv=cv5,
                               scoring='roc_auc', random_state=42, n_jobs=-1)
xgb_rs.fit(X_bal_m, y_bal_m)
xgb_m = xgb_rs.best_estimator_
evaluate('XGBoost (MIMIC)', xgb_m, X_test_m, y_test_m, 'MIMIC')
print(f"  Best params: {xgb_rs.best_params_}")
joblib.dump(xgb_m, os.path.join(MODEL_DIR, 'xgb_mimic.pkl'))

# ── 8D · LightGBM (MIMIC) ────────────────────────────────
print("\n[MIMIC] LightGBM …")
lgb_param = dict(n_estimators=[200,400], num_leaves=[31,63],
                  learning_rate=[0.05,0.1], min_child_samples=[10,20])
lgb_base = lgb.LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1)
lgb_rs   = RandomizedSearchCV(lgb_base, lgb_param, n_iter=8, cv=cv5,
                               scoring='roc_auc', random_state=42, n_jobs=-1)
lgb_rs.fit(X_bal_m, y_bal_m)
lgb_m = lgb_rs.best_estimator_
evaluate('LightGBM (MIMIC)', lgb_m, X_test_m, y_test_m, 'MIMIC')
print(f"  Best params: {lgb_rs.best_params_}")
joblib.dump(lgb_m, os.path.join(MODEL_DIR, 'lgb_mimic.pkl'))

# ── 8E · LR on external Vitals dataset ───────────────────
print("\n[VITALS] Logistic Regression …")
lr_v = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr_v.fit(X_bal_v, y_bal_v)
evaluate('LR (Vitals)', lr_v, X_test_v, y_test_v, 'Vitals')
joblib.dump(lr_v, os.path.join(MODEL_DIR, 'lr_vitals.pkl'))

# ── 8F · XGBoost on external Vitals ──────────────────────
print("\n[VITALS] XGBoost …")
xgb_v = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.08,
                            subsample=0.85, colsample_bytree=0.85,
                            use_label_encoder=False, eval_metric='logloss',
                            random_state=42, n_jobs=-1)
xgb_v.fit(X_bal_v, y_bal_v)
evaluate('XGBoost (Vitals)', xgb_v, X_test_v, y_test_v, 'Vitals')
joblib.dump(xgb_v, os.path.join(MODEL_DIR, 'xgb_vitals.pkl'))

# ── 8G · LightGBM on external Vitals ─────────────────────
print("\n[VITALS] LightGBM …")
lgb_v = lgb.LGBMClassifier(n_estimators=300, num_leaves=63, learning_rate=0.08,
                             random_state=42, n_jobs=-1, verbose=-1)
lgb_v.fit(X_bal_v, y_bal_v)
evaluate('LightGBM (Vitals)', lgb_v, X_test_v, y_test_v, 'Vitals')
joblib.dump(lgb_v, os.path.join(MODEL_DIR, 'lgb_vitals.pkl'))

# ── 8H · Cox PH Survival Model (MIMIC) ───────────────────
print("\n[MIMIC] Cox Proportional Hazards …")
try:
    surv_cols = [c for c in feature_cols if c in df.columns]
    df_surv2 = df[surv_cols + ['los_hours','high_uncertainty']].copy()
    df_surv2['duration'] = df_surv2['los_hours'].fillna(df_surv2['los_hours'].median())
    df_surv2 = df_surv2[df_surv2['duration'] > 0].copy()
    df_surv2_imp = pd.DataFrame(imputer.transform(df_surv2[surv_cols]), columns=surv_cols)
    df_surv2_imp['duration'] = df_surv2['duration'].values
    df_surv2_imp['event']    = df_surv2['high_uncertainty'].values
    df_surv2_imp = df_surv2_imp.replace([np.inf, -np.inf], np.nan).fillna(df_surv2_imp.median())

    # Use only top features for Cox (avoid singularity)
    cox_feats = surv_cols[:15]
    cox_df    = df_surv2_imp[cox_feats + ['duration','event']]
    cph = CoxPHFitter(penalizer=0.1)
    cph.fit(cox_df, duration_col='duration', event_col='event')
    c_index = cph.concordance_index_
    print(f"  C-Index: {c_index:.4f}")
    results['Cox PH (MIMIC)'] = dict(auc=None, ap=None, f1=None,
                                      c_index=c_index, dataset='MIMIC')
    with open(os.path.join(MODEL_DIR, 'coxph_mimic.pkl'), 'wb') as fh:
        pickle.dump(cph, fh)
except Exception as e:
    print(f"  ⚠️  Cox PH skipped: {e}")

print("\n✅ All models trained")



════════════════════════════════════════════════════════════
  SECTION 8 · Model Training
════════════════════════════════════════════════════════════

[MIMIC] Logistic Regression …
  LR (MIMIC)                          AUC=0.9333  AP=0.8842  F1=0.8182
  CV-5 AUC: 0.9417

[MIMIC] Random Forest …
  RF (MIMIC)                          AUC=0.8867  AP=0.8225  F1=0.7200
  Best params: {'n_estimators': 400, 'min_samples_split': 2, 'max_features': 'log2', 'max_depth': 10}

[MIMIC] XGBoost …
  XGBoost (MIMIC)                     AUC=0.8333  AP=0.7245  F1=0.6957
  Best params: {'subsample': 1.0, 'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 1.0}

[MIMIC] LightGBM …
  LightGBM (MIMIC)                    AUC=0.8600  AP=0.7881  F1=0.7826
  Best params: {'num_leaves': 31, 'n_estimators': 200, 'min_child_samples': 10, 'learning_rate': 0.05}

[VITALS] Logistic Regression …
  LR (Vitals)                         AUC=0.8996  AP=0.9210  F1=0.8245

[VITALS] XGBoost …
  XG

In [ ]:
# ══════════════════════════════════════════════════════════════
# SECTION 9 — ENSEMBLE
# ══════════════════════════════════════════════════════════════
print("\n" + "═"*60)
print("  SECTION 9 · Ensemble")
print("═"*60)

# Soft voting ensemble on MIMIC test set
mimic_models = [
    ('LR',       lr),
    ('RF',       rf),
    ('XGBoost',  xgb_m),
    ('LightGBM', lgb_m),
]
ens_probs = np.mean(
    [m.predict_proba(X_test_m)[:, 1] for _, m in mimic_models], axis=0)
ens_pred  = (ens_probs >= 0.5).astype(int)
ens_auc   = roc_auc_score(y_test_m, ens_probs)
ens_ap    = average_precision_score(y_test_m, ens_probs)
ens_f1    = f1_score(y_test_m, ens_pred)

results['Ensemble (MIMIC)'] = dict(auc=ens_auc, ap=ens_ap, f1=ens_f1,
                                    y_prob=ens_probs, y_pred=ens_pred,
                                    y_test=np.array(y_test_m), dataset='MIMIC')
print(f"  Ensemble (MIMIC)  AUC={ens_auc:.4f}  AP={ens_ap:.4f}  F1={ens_f1:.4f}")

# Vitals ensemble
vitals_models = [('LR_v', lr_v), ('XGB_v', xgb_v), ('LGB_v', lgb_v)]
ens_v_probs   = np.mean([m.predict_proba(X_test_v)[:,1] for _,m in vitals_models], axis=0)
ens_v_pred    = (ens_v_probs >= 0.5).astype(int)
ens_v_auc     = roc_auc_score(y_test_v, ens_v_probs)
ens_v_f1      = f1_score(y_test_v, ens_v_pred)
results['Ensemble (Vitals)'] = dict(auc=ens_v_auc, ap=average_precision_score(y_test_v, ens_v_probs),
                                     f1=ens_v_f1, y_prob=ens_v_probs, y_pred=ens_v_pred,
                                     y_test=np.array(y_test_v), dataset='Vitals')
print(f"  Ensemble (Vitals) AUC={ens_v_auc:.4f}  F1={ens_v_f1:.4f}")


════════════════════════════════════════════════════════════
  SECTION 9 · Ensemble
════════════════════════════════════════════════════════════
  Ensemble (MIMIC)  AUC=0.8933  AP=0.8175  F1=0.8333
  Ensemble (Vitals) AUC=0.9999  F1=0.9979


In [ ]:
print("\n" + "═"*60)
print("  SECTION 10 · Evaluation Plots")
print("═"*60)

# Separate MIMIC and Vitals results for plotting
mimic_res  = {k:v for k,v in results.items() if v.get('dataset')=='MIMIC' and v['auc']}
vitals_res = {k:v for k,v in results.items() if v.get('dataset')=='Vitals' and v['auc']}
all_cls    = {**mimic_res, **vitals_res}
model_colors = {
    'LR (MIMIC)'         : TEAL,
    'RF (MIMIC)'         : BLUE,
    'XGBoost (MIMIC)'    : PURPLE,
    'LightGBM (MIMIC)'   : GREEN,
    'Ensemble (MIMIC)'   : RED,
    'LR (Vitals)'        : '#a3e635',
    'XGBoost (Vitals)'   : '#f472b6',
    'LightGBM (Vitals)'  : AMBER,
    'Ensemble (Vitals)'  : '#fb923c',
}

# ── Plot 6 · ROC Curves ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('ROC Curves — All Models', fontweight='bold')

for ax, res_dict, title in [
    (axes[0], mimic_res,  'MIMIC-IV Dataset'),
    (axes[1], vitals_res, 'External Vitals Dataset')
]:
    ax.plot([0,1],[0,1],'--', color='#475569', lw=1, alpha=0.6, label='Random')
    for name, r in res_dict.items():
        fpr, tpr, _ = roc_curve(r['y_test'], r['y_prob'])
        ax.plot(fpr, tpr, lw=2, color=model_colors.get(name, BLUE),
                label=f"{name.split('(')[0].strip()} (AUC={r['auc']:.3f})")
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(title); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout(); savefig('06_roc_curves.png')

# ── Plot 7 · Precision-Recall Curves ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Precision-Recall Curves — All Models', fontweight='bold')

for ax, res_dict, title in [
    (axes[0], mimic_res,  'MIMIC-IV Dataset'),
    (axes[1], vitals_res, 'External Vitals Dataset')
]:
    for name, r in res_dict.items():
        prec, rec, _ = precision_recall_curve(r['y_test'], r['y_prob'])
        ax.plot(rec, prec, lw=2, color=model_colors.get(name, BLUE),
                label=f"{name.split('(')[0].strip()} (AP={r['ap']:.3f})")
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title(title); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout(); savefig('07_precision_recall_curves.png')

# ── Plot 8 · Confusion Matrices ──────────────────────────
mimic_cls  = [(k,v) for k,v in mimic_res.items()  if k != 'Ensemble (MIMIC)']
vitals_cls = [(k,v) for k,v in vitals_res.items() if k != 'Ensemble (Vitals)']
plot_cms   = mimic_cls + vitals_cls

n_cm = len(plot_cms)
cols = 4; rows = (n_cm + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*4))
fig.suptitle('Confusion Matrices — All Models', fontweight='bold')
axes_flat  = axes.flatten() if n_cm > 1 else [axes]

for ax, (name, r) in zip(axes_flat, plot_cms):
    cm   = confusion_matrix(r['y_test'], r['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Low','High'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    short = name.split('(')[0].strip()
    ax.set_title(f"{short}\nAUC={r['auc']:.3f}", fontsize=9,
                 color=model_colors.get(name, BLUE))

for ax in axes_flat[n_cm:]:
    ax.set_visible(False)

plt.tight_layout(); savefig('08_confusion_matrices.png')

# ── Plot 9 · AUC Leaderboard Bar ─────────────────────────
lb_data = [(k, v['auc'], v['f1'], v['dataset'])
           for k,v in results.items() if v.get('auc')]
lb_data.sort(key=lambda x: x[1])

names  = [x[0].replace(' (MIMIC)','').replace(' (Vitals)','') + '\n(' + x[3] + ')' for x in lb_data]
aucs   = [x[1] for x in lb_data]
f1s    = [x[2] for x in lb_data]
colors_bar = [RED if 'Ensemble' in x[0] else (BLUE if x[3]=='MIMIC' else AMBER) for x in lb_data]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Model Leaderboard — AUC-ROC & F1 Score', fontweight='bold')

for ax, vals, xlabel, title in [
    (axes[0], aucs, 'AUC-ROC', 'AUC-ROC Leaderboard'),
    (axes[1], f1s,  'F1 Score', 'F1 Score Leaderboard')
]:
    bars = ax.barh(names, vals, color=colors_bar, edgecolor='#30363d', linewidth=0.5)
    for bar, val in zip(bars, vals):
        ax.text(val + 0.003, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=8.5)
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    ax.set_xlim(0, min(1.05, max(vals) + 0.07))
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout(); savefig('09_leaderboard.png')

# ── Plot 10 · Calibration Curves ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Calibration Curves — Predicted vs True Probability', fontweight='bold')

for ax, res_dict, title in [
    (axes[0], mimic_res,  'MIMIC-IV Dataset'),
    (axes[1], vitals_res, 'External Vitals Dataset')
]:
    ax.plot([0,1],[0,1],'--', color='#475569', lw=1.5, label='Perfect calibration')
    for name, r in res_dict.items():
        fp, mp = calibration_curve(r['y_test'], r['y_prob'], n_bins=8)
        ax.plot(mp, fp, marker='o', lw=1.5, color=model_colors.get(name, BLUE),
                label=name.split('(')[0].strip())
    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Fraction of Positives')
    ax.set_title(title); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout(); savefig('10_calibration_curves.png')

# ── Plot 11 · Score Distributions ────────────────────────
top_models = ['RF (MIMIC)', 'XGBoost (MIMIC)', 'Ensemble (MIMIC)',
              'XGBoost (Vitals)', 'Ensemble (Vitals)']
top_models = [m for m in top_models if m in results and results[m].get('y_prob') is not None]

fig, axes = plt.subplots(1, len(top_models), figsize=(4*len(top_models), 4))
fig.suptitle('Predicted Score Distributions — Class Separation', fontweight='bold')
if len(top_models) == 1: axes = [axes]

for ax, name in zip(axes, top_models):
    r = results[name]
    lo = r['y_prob'][r['y_test']==0]
    hi = r['y_prob'][r['y_test']==1]
    ax.hist(lo, bins=20, color=BLUE, alpha=0.65, density=True, label='Low')
    ax.hist(hi, bins=20, color=RED,  alpha=0.65, density=True, label='High')
    ax.axvline(0.5, color=AMBER, lw=1.5, ls='--')
    ax.set_title(name.replace(' (MIMIC)','').replace(' (Vitals)',''), fontsize=9)
    ax.set_xlabel('Score'); ax.legend(fontsize=7)

plt.tight_layout(); savefig('11_score_distributions.png')

print("✅ Evaluation plots saved")


════════════════════════════════════════════════════════════
  SECTION 10 · Evaluation Plots
════════════════════════════════════════════════════════════
  📊 Saved: 06_roc_curves.png
  📊 Saved: 07_precision_recall_curves.png
  📊 Saved: 08_confusion_matrices.png
  📊 Saved: 09_leaderboard.png
  📊 Saved: 10_calibration_curves.png
  📊 Saved: 11_score_distributions.png
✅ Evaluation plots saved


In [ ]:
print("\n" + "═"*60)
print("  SECTION 11 · SHAP Feature Importance")
print("═"*60)

# ── SHAP for XGBoost (MIMIC) ─────────────────────────────
print("⏳ Computing SHAP values for XGBoost (MIMIC) …")
try:
    explainer_xgb = shap.TreeExplainer(xgb_m)
    shap_vals_xgb = explainer_xgb.shap_values(X_test_m)

    # Summary bar plot
    fig, ax = plt.subplots(figsize=(10, 7))
    shap.summary_plot(shap_vals_xgb, X_test_m, plot_type='bar',
                      max_display=18, show=False)
    plt.title('SHAP Feature Importance — XGBoost (MIMIC)', fontweight='bold', pad=10)
    plt.tight_layout(); savefig('12_shap_summary_bar_xgb.png')

    # SHAP beeswarm
    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(shap_vals_xgb, X_test_m, max_display=18, show=False)
    plt.title('SHAP Beeswarm — XGBoost (MIMIC)', fontweight='bold', pad=10)
    plt.tight_layout(); savefig('13_shap_beeswarm_xgb.png')

    # Waterfall for highest-uncertainty patient
    high_idx   = np.argmax(results['XGBoost (MIMIC)']['y_prob'])
    shap_exp   = shap.Explanation(
        values=shap_vals_xgb[high_idx],
        base_values=explainer_xgb.expected_value,
        data=X_test_m.iloc[high_idx].values,
        feature_names=list(X_test_m.columns)
    )
    fig, ax = plt.subplots(figsize=(10, 7))
    shap.waterfall_plot(shap_exp, max_display=15, show=False)
    plt.title(f'SHAP Waterfall — Highest-Uncertainty Patient (idx={high_idx})',
              fontweight='bold', pad=10)
    plt.tight_layout(); savefig('14_shap_waterfall_high_unc.png')

except Exception as e:
    print(f"  ⚠️  SHAP skipped: {e}")

# ── SHAP for XGBoost (Vitals) ─────────────────────────────
print("⏳ Computing SHAP values for XGBoost (Vitals) …")
try:
    explainer_v = shap.TreeExplainer(xgb_v)
    shap_vals_v = explainer_v.shap_values(X_test_v.iloc[:2000])  # subset for speed

    fig, ax = plt.subplots(figsize=(10, 6))
    shap.summary_plot(shap_vals_v, X_test_v.iloc[:2000], plot_type='bar',
                      max_display=15, show=False)
    plt.title('SHAP Feature Importance — XGBoost (Vitals Dataset)', fontweight='bold', pad=10)
    plt.tight_layout(); savefig('15_shap_summary_vitals.png')
except Exception as e:
    print(f"  ⚠️  SHAP Vitals skipped: {e}")

print("✅ SHAP plots saved")


════════════════════════════════════════════════════════════
  SECTION 11 · SHAP Feature Importance
════════════════════════════════════════════════════════════
⏳ Computing SHAP values for XGBoost (MIMIC) …
  📊 Saved: 12_shap_summary_bar_xgb.png
  📊 Saved: 13_shap_beeswarm_xgb.png
  📊 Saved: 14_shap_waterfall_high_unc.png
⏳ Computing SHAP values for XGBoost (Vitals) …
  📊 Saved: 15_shap_summary_vitals.png
✅ SHAP plots saved


In [ ]:
print("\n" + "═"*60)
print("  SECTION 12 · Feature Importance Consensus")
print("═"*60)

try:
    rf_imp  = pd.Series(rf.feature_importances_,  index=feature_cols)
    xgb_imp = pd.Series(xgb_m.feature_importances_, index=feature_cols)
    lgb_imp = pd.Series(lgb_m.feature_importances_, index=feature_cols)

    rf_imp  = rf_imp  / (rf_imp.sum()  + 1e-9)
    xgb_imp = xgb_imp / (xgb_imp.sum()+ 1e-9)
    lgb_imp = lgb_imp / (lgb_imp.sum()+ 1e-9)

    avg_imp   = (rf_imp + xgb_imp + lgb_imp) / 3
    top_feats = avg_imp.nlargest(18).index.tolist()

    imp_df = pd.DataFrame({
        'Random Forest' : rf_imp[top_feats].values,
        'XGBoost'       : xgb_imp[top_feats].values,
        'LightGBM'      : lgb_imp[top_feats].values,
    }, index=top_feats)

    fig, ax = plt.subplots(figsize=(12, 8))
    x     = np.arange(len(top_feats))
    width = 0.26
    ax.barh(x + width, imp_df['Random Forest'], width, label='Random Forest',color=BLUE,  alpha=0.85)
    ax.barh(x,         imp_df['XGBoost'],       width, label='XGBoost',      color=PURPLE, alpha=0.85)
    ax.barh(x - width, imp_df['LightGBM'],      width, label='LightGBM',     color=GREEN,  alpha=0.85)
    ax.set_yticks(x); ax.set_yticklabels(top_feats, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Normalised Importance')
    ax.set_title('Feature Importance Consensus — RF vs XGBoost vs LightGBM (MIMIC)',
                 fontweight='bold')
    ax.legend(); ax.grid(axis='x', alpha=0.3)
    plt.tight_layout(); savefig('16_feature_importance_consensus.png')
except Exception as e:
    print(f"  ⚠️  Feature importance plot skipped: {e}")


════════════════════════════════════════════════════════════
  SECTION 12 · Feature Importance Consensus
════════════════════════════════════════════════════════════
  📊 Saved: 16_feature_importance_consensus.png


In [ ]:
print("\n" + "═"*60)
print("  SECTION 13 · Cox PH Visualisations")
print("═"*60)

if 'Cox PH (MIMIC)' in results:
    try:
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        fig.suptitle('Cox Proportional Hazards — MIMIC-IV', fontweight='bold')

        cph.plot(ax=axes[0])
        axes[0].set_title(f'Hazard Ratios (C-Index={c_index:.4f})')
        axes[0].axvline(0, color=RED, ls='--', lw=1)
        axes[0].grid(True, alpha=0.3)

        cph.baseline_survival_.plot(ax=axes[1], color=BLUE, lw=2, legend=False)
        axes[1].set_title('Baseline Survival Function')
        axes[1].set_xlabel('ICU Hours'); axes[1].set_ylabel('Survival Probability')
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout(); savefig('17_coxph_plots.png')
    except Exception as e:
        print(f"  ⚠️  Cox PH plots skipped: {e}")


════════════════════════════════════════════════════════════
  SECTION 13 · Cox PH Visualisations
════════════════════════════════════════════════════════════


In [ ]:
print("\n" + "═"*60)
print("  SECTION 14 · Results Summary")
print("═"*60)

rows = []
for name, r in results.items():
    rows.append({
        'Model'   : name,
        'Dataset' : r.get('dataset', '—'),
        'AUC-ROC' : round(r['auc'], 4)     if r.get('auc')     else '—',
        'Avg Prec': round(r['ap'],  4)     if r.get('ap')      else '—',
        'F1'      : round(r['f1'],  4)     if r.get('f1')      else '—',
        'C-Index' : round(r['c_index'], 4) if r.get('c_index') else '—',
    })

results_df = pd.DataFrame(rows)
print("\n" + "="*72)
print("          FULL MODEL LEADERBOARD")
print("="*72)
print(results_df.to_string(index=False))
print("="*72)

# Identify best
auc_rows = results_df[results_df['AUC-ROC'] != '—'].copy()
auc_rows['AUC-ROC'] = auc_rows['AUC-ROC'].astype(float)
best_row = auc_rows.loc[auc_rows['AUC-ROC'].idxmax()]
print(f"\n🏆  BEST MODEL : {best_row['Model']}  |  AUC = {best_row['AUC-ROC']:.4f}")

# Save CSV
csv_path = os.path.join(RESULT_DIR, 'full_leaderboard.csv')
results_df.to_csv(csv_path, index=False)
print(f"\n✅  Results CSV saved : {csv_path}")

# Save full results dict (probabilities, ground truth etc.)
with open(os.path.join(RESULT_DIR, 'results_dict.pkl'), 'wb') as fh:
    # strip large arrays before saving; keep metrics only
    meta = {k: {kk:vv for kk,vv in v.items()
                if kk not in ['y_prob','y_pred','y_test']}
            for k,v in results.items()}
    pickle.dump(meta, fh)


════════════════════════════════════════════════════════════
  SECTION 14 · Results Summary
════════════════════════════════════════════════════════════

          FULL MODEL LEADERBOARD
            Model Dataset  AUC-ROC  Avg Prec     F1 C-Index
       LR (MIMIC)   MIMIC   0.9333    0.8842 0.8182       —
       RF (MIMIC)   MIMIC   0.8867    0.8225 0.7200       —
  XGBoost (MIMIC)   MIMIC   0.8333    0.7245 0.6957       —
 LightGBM (MIMIC)   MIMIC   0.8600    0.7881 0.7826       —
      LR (Vitals)  Vitals   0.8996    0.9210 0.8245       —
 XGBoost (Vitals)  Vitals   1.0000    1.0000 0.9970       —
LightGBM (Vitals)  Vitals   1.0000    1.0000 0.9983       —
 Ensemble (MIMIC)   MIMIC   0.8933    0.8175 0.8333       —
Ensemble (Vitals)  Vitals   0.9999    0.9999 0.9979       —

🏆  BEST MODEL : XGBoost (Vitals)  |  AUC = 1.0000

✅  Results CSV saved : /content/drive/MyDrive/CUEPE_Outputs/results/full_leaderboard.csv


In [ ]:
print("\n" + "═"*60)
print("  SECTION 15 · Sample Clinical Uncertainty Score Inference")
print("═"*60)

print("\n  ── MIMIC-IV test set (first 8 patients) ──")
sample = X_test_m.head(8)
probs_ens  = np.mean([m.predict_proba(sample)[:,1] for _,m in mimic_models], axis=0)

for i, (prob, true_label) in enumerate(
        zip(probs_ens, y_test_m.head(8).values)):
    if prob >= 0.65:
        flag = '🔴 ESCALATE'
    elif prob >= 0.35:
        flag = '🟡 WATCH'
    else:
        flag = '🟢 MONITOR'
    actual = 'HIGH' if true_label==1 else 'LOW'
    print(f"  Patient {i+1:02d} | CUI={prob:.3f} | {flag} | Actual: {actual}")


════════════════════════════════════════════════════════════
  SECTION 15 · Sample Clinical Uncertainty Score Inference
════════════════════════════════════════════════════════════

  ── MIMIC-IV test set (first 8 patients) ──
  Patient 01 | CUI=0.938 | 🔴 ESCALATE | Actual: HIGH
  Patient 02 | CUI=0.037 | 🟢 MONITOR | Actual: LOW
  Patient 03 | CUI=0.547 | 🟡 WATCH | Actual: HIGH
  Patient 04 | CUI=0.097 | 🟢 MONITOR | Actual: LOW
  Patient 05 | CUI=0.046 | 🟢 MONITOR | Actual: LOW
  Patient 06 | CUI=0.965 | 🔴 ESCALATE | Actual: HIGH
  Patient 07 | CUI=0.015 | 🟢 MONITOR | Actual: LOW
  Patient 08 | CUI=0.855 | 🔴 ESCALATE | Actual: HIGH


In [ ]:
print("\n" + "═"*60)
print("  FINAL · Saved Files")
print("═"*60)
for root, dirs, files in os.walk(BASE_OUT):
    for fn in sorted(files):
        fp   = os.path.join(root, fn)
        size = os.path.getsize(fp) / 1024
        tag  = '🧠' if fn.endswith('.pkl') else '📊'
        rel  = os.path.relpath(fp, BASE_OUT)
        print(f"  {tag}  {rel:55s} ({size:.1f} KB)")

print("\n🎉  CUEPE Pipeline complete!")
print("""
╔══════════════════════════════════════════════════════════╗
║  DATASETS USED                                           ║
║  ├─ MIMIC-IV Demo  →  icu/chartevents.csv               ║
║  │                     icu/icustays.csv                  ║
║  │                     hosp/patients.csv                 ║
║  │                     hosp/admissions.csv               ║
║  │                     hosp/labevents.csv                ║
║  └─ External       →  human_vital_signs_dataset_2024.csv ║
╠══════════════════════════════════════════════════════════╣
║  MODELS TRAINED                                          ║
║  ├─ Logistic Regression   (MIMIC + Vitals)               ║
║  ├─ Random Forest         (MIMIC)                        ║
║  ├─ XGBoost               (MIMIC + Vitals)               ║
║  ├─ LightGBM              (MIMIC + Vitals)               ║
║  ├─ Cox Proportional Hazards (MIMIC survival)            ║
║  └─ Soft-Voting Ensemble  (MIMIC + Vitals)               ║
╠══════════════════════════════════════════════════════════╣
║  PLOTS GENERATED  (outputs/plots/)                       ║
║  01 CUI Distribution          10 Calibration Curves      ║
║  02 Vital Entropy by Class    11 Score Distributions     ║
║  03 Correlation Heatmap       12 SHAP Summary Bar        ║
║  04 LOS by Uncertainty        13 SHAP Beeswarm           ║
║  05 Kaplan-Meier Survival     14 SHAP Waterfall          ║
║  06 ROC Curves                15 SHAP Vitals Summary     ║
║  07 Precision-Recall Curves   16 Feature Importance      ║
║  08 Confusion Matrices        17 Cox PH Hazard Ratios    ║
║  09 Leaderboard Bar Chart                                 ║
╚══════════════════════════════════════════════════════════╝
""")


════════════════════════════════════════════════════════════
  FINAL · Saved Files
════════════════════════════════════════════════════════════
  📊  plots/01_cui_distribution.png                           (69.5 KB)
  📊  plots/02_vital_entropy_by_class.png                     (52.7 KB)
  📊  plots/03_correlation_heatmap.png                        (128.7 KB)
  📊  plots/04_los_by_uncertainty.png                         (66.6 KB)
  📊  plots/05_kaplan_meier.png                               (50.2 KB)
  📊  plots/06_roc_curves.png                                 (109.3 KB)
  📊  plots/07_precision_recall_curves.png                    (125.3 KB)
  📊  plots/08_confusion_matrices.png                         (88.7 KB)
  📊  plots/09_leaderboard.png                                (108.0 KB)
  📊  plots/10_calibration_curves.png                         (241.5 KB)
  📊  plots/11_score_distributions.png                        (49.4 KB)
  📊  plots/12_shap_summary_bar_xgb.png                       (97.5 KB